In [ ]:
!pip install -q boruta umap-learn

import os
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, silhouette_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from boruta import BorutaPy
import umap

# Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
INPUT_FILE = Path("bayesiannetwork_150_edited.csv")
OUTPUT_DIR = Path("outputs")
REPORT_DIR = OUTPUT_DIR / "reports"
PLOT_DIR = OUTPUT_DIR / "plots"
TABLE_DIR = OUTPUT_DIR / "tables"

for directory in [OUTPUT_DIR, REPORT_DIR, PLOT_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def normalize_name(name):
    return str(name).strip().lower().replace(" ", "_").replace("-", "_").replace("/", "_").replace("(", "").replace(")", "")

def find_column(df, candidates):
    normalized = {normalize_name(c): c for c in df.columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized: return normalized[key]
    return None

def prepare_xy(df, feature_cols, target_col):
    X_raw, y_raw = df[feature_cols].copy(), df[target_col].copy()
    target_encoder = LabelEncoder()
    y = target_encoder.fit_transform(y_raw.astype(str).fillna("__missing__"))
    numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(X_raw[c])]
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]
    preprocessor = ColumnTransformer(transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]), categorical_cols)
    ])
    X = pd.DataFrame(preprocessor.fit_transform(X_raw), columns=preprocessor.get_feature_names_out(), index=df.index).astype(float)
    return X, y, target_encoder

# Execution logic starts here
if not INPUT_FILE.exists():
    print(f"Please upload {INPUT_FILE} to run this pipeline.")
else:
    df = pd.read_csv(INPUT_FILE)
    diagnosis_cols = [c for c in df.columns if normalize_name(c) == "diagnosis"]
    df = df.drop(columns=diagnosis_cols)
    target_col = find_column(df, ["Target_Class"])

    if target_col:
        X, y, encoder = prepare_xy(df, [c for c in df.columns if c != target_col], target_col)
        print("Data loaded and preprocessed. Ready for analysis.")
    else:
        print("Target_Class not found.")

In [ ]:
# STEP 2: Feature Selection & Ranking
# 2A: Correlation Analysis
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]
X_filtered = X.drop(columns=to_drop)

# 2B: Mutual Information
mi_scores = mutual_info_classif(X_filtered, y, random_state=RANDOM_SEED)
mi_df = pd.DataFrame({'Feature': X_filtered.columns, 'MI_Score': mi_scores}).sort_values('MI_Score', ascending=False)

# 2C: Boruta Selection
rf = RandomForestClassifier(n_jobs=-1, class_weight='balanced', max_depth=5, random_state=RANDOM_SEED)
boruta_selector = BorutaPy(rf, n_estimators='auto', verbose=0, random_state=RANDOM_SEED)
boruta_selector.fit(X_filtered.values, y)

boruta_results = pd.DataFrame({
    'Feature': X_filtered.columns,
    'Status': np.where(boruta_selector.support_, 'Confirmed', np.where(boruta_selector.support_weak_, 'Tentative', 'Rejected')),
    'Ranking': boruta_selector.ranking_
}).sort_values('Ranking')

print("Top Features by Mutual Information:")
display(mi_df.head(10))
print("\nBoruta Selection Results:")
display(boruta_results[boruta_results['Status'] != 'Rejected'])

# Plotting MI
plt.figure(figsize=(10, 6))
sns.barplot(data=mi_df.head(15), x='MI_Score', y='Feature', palette='viridis')
plt.title('Top 15 Features by Mutual Information')
plt.show()

In [ ]:
# STEP 3: Performance Comparison (ECG vs. Physiological vs. Combined)

# Define feature groups based on confirmed/retained features
confirmed_ecg = boruta_results[boruta_results['Status'] == 'Confirmed']['Feature'].tolist()
phys_features = [f for f in X_filtered.columns if 'BMI' in f or 'Cycle' in f or 'History' in f]

feature_groups = {
    "ECG Only (Confirmed)": confirmed_ecg,
    "Physiological Only": phys_features,
    "Combined Set": list(set(confirmed_ecg + phys_features))
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
results = []

for name, cols in feature_groups.items():
    if not cols: continue

    fold_metrics = []
    for train_idx, test_idx in cv.split(X_filtered[cols], y):
        model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_SEED)
        model.fit(X_filtered[cols].iloc[train_idx], y[train_idx])
        y_pred = model.predict(X_filtered[cols].iloc[test_idx])
        y_prob = model.predict_proba(X_filtered[cols].iloc[test_idx])[:, 1]

        fold_metrics.append({
            'Accuracy': accuracy_score(y[test_idx], y_pred),
            'F1': f1_score(y[test_idx], y_pred),
            'AUC': roc_auc_score(y[test_idx], y_prob)
        })

    avg_metrics = pd.DataFrame(fold_metrics).mean()
    results.append({'Group': name, **avg_metrics.to_dict(), 'Feature Count': len(cols)})

comparison_df = pd.DataFrame(results)
print("Cross-Validation Performance Comparison:")
display(comparison_df)

# Visualize Comparison
plt.figure(figsize=(10, 5))
plot_data = comparison_df.melt(id_vars='Group', value_vars=['Accuracy', 'F1', 'AUC'], var_name='Metric', value_name='Score')
sns.barplot(data=plot_data, x='Metric', y='Score', hue='Group')
plt.title('Performance by Feature Subset (5-Fold CV)')
plt.ylim(0, 1.0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### STEP 4: ECG-Centric Weighted Ensemble
To give higher importance to ECG features, we implement a Weighted Ensemble (Soft Voting). We assign a higher weight to the ECG sub-model to ensure the screening system prioritizes cardiovascular biomarkers over self-reported physiological data.

In [ ]:
from sklearn.ensemble import VotingClassifier

# Define weights: 0.7 for ECG, 0.3 for Physiological
weights = [0.7, 0.3]

# Initialize sub-models
ecg_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_SEED)
phys_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_SEED)

# Cross-validation for the Weighted Ensemble
ensemble_metrics = []

for train_idx, test_idx in cv.split(X_filtered, y):
    # Split data
    X_train, X_test = X_filtered.iloc[train_idx], X_filtered.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Train sub-models on their respective feature sets
    ecg_model.fit(X_train[confirmed_ecg], y_train)
    phys_model.fit(X_train[phys_features], y_train)

    # Get probabilities
    prob_ecg = ecg_model.predict_proba(X_test[confirmed_ecg])[:, 1]
    prob_phys = phys_model.predict_proba(X_test[phys_features])[:, 1]

    # Weighted combination of probabilities
    final_prob = (prob_ecg * weights[0]) + (prob_phys * weights[1])
    final_pred = (final_prob > 0.5).astype(int)

    ensemble_metrics.append({
        'Accuracy': accuracy_score(y_test, final_pred),
        'F1': f1_score(y_test, final_pred),
        'AUC': roc_auc_score(y_test, final_prob)
    })

weighted_results = pd.DataFrame(ensemble_metrics).mean()
print("ECG-Centric Weighted Ensemble Results (70/30 Weighting):")
display(weighted_results.to_frame().T)

# Compare with previous results
comparison_extended = pd.concat([comparison_df, pd.DataFrame([{
    'Group': 'ECG-Weighted Ensemble',
    'Accuracy': weighted_results['Accuracy'],
    'F1': weighted_results['F1'],
    'AUC': weighted_results['AUC'],
    'Feature Count': len(confirmed_ecg) + len(phys_features)
}])], ignore_index=True)

display(comparison_extended)

### STEP 5: Hyperparameter Tuning for Optimization
We will use Grid Search to optimize the Random Forest parameters (depth, estimators, and leaf size) specifically for our ECG-centric approach.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced']
}

print("Optimizing ECG sub-model...")
grid_ecg = GridSearchCV(RandomForestClassifier(random_state=RANDOM_SEED), param_grid, cv=5, scoring='roc_auc')
grid_ecg.fit(X_filtered[confirmed_ecg], y)

print("Optimizing Physiological sub-model...")
grid_phys = GridSearchCV(RandomForestClassifier(random_state=RANDOM_SEED), param_grid, cv=5, scoring='roc_auc')
grid_phys.fit(X_filtered[phys_features], y)

best_ecg = grid_ecg.best_estimator_
best_phys = grid_phys.best_estimator_

print(f"Best ECG Params: {grid_ecg.best_params_}")
print(f"Best Phys Params: {grid_phys.best_params_}")

### STEP 6: Final Model Export
Now we save the trained ensemble and the feature lists. This is essential for the 'External Validation' task.

In [ ]:
import joblib

# Final fit on entire dataset
best_ecg.fit(X_filtered[confirmed_ecg], y)
best_phys.fit(X_filtered[phys_features], y)

model_payload = {
    'ecg_model': best_ecg,
    'phys_model': best_phys,
    'ecg_features': confirmed_ecg,
    'phys_features': phys_features,
    'weights': [0.7, 0.3]
}

joblib.dump(model_payload, 'pcos_screening_model.joblib')
print("Model saved as 'pcos_screening_model.joblib'")

### STEP 7: External Validation Script
This script demonstrates how to load the model and apply it to a new dataset (`external_cohort.csv`).

In [ ]:
def run_external_validation(new_data_path):
    try:
        # Load model
        artifacts = joblib.load('pcos_screening_model.joblib')
        new_df = pd.read_csv(new_data_path)

        # In a real scenario, you would apply the same preprocessing pipeline here
        # For this demo, we assume columns are already named/scaled correctly
        X_new = new_df

        p_ecg = artifacts['ecg_model'].predict_proba(X_new[artifacts['ecg_features']])[:, 1]
        p_phys = artifacts['phys_model'].predict_proba(X_new[artifacts['phys_features']])[:, 1]

        final_scores = (p_ecg * artifacts['weights'][0]) + (p_phys * artifacts['weights'][1])

        print(f"Validation complete for {len(new_df)} samples.")
        return final_scores
    except Exception as e:
        return f"Validation failed: {str(e)}"

print("External validation function ready.")

### STEP 8: Dimensionality Reduction & Visualization
We evaluate if ECG features can visually separate PCOS from Normal using PCA, UMAP, and t-SNE.

In [ ]:
import umap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Standardize data for projection
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

# 1. PCA
pca = PCA(n_components=2)
pca_res = pca.fit_transform(X_scaled)

# 2. UMAP
umap_mod = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=RANDOM_SEED)
umap_res = umap_mod.fit_transform(X_scaled)

# 3. t-SNE
tsne = TSNE(n_components=2, random_state=RANDOM_SEED)
tsne_res = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['PCA Projection', 'UMAP Projection', 't-SNE Projection']
res_list = [pca_res, umap_res, tsne_res]

for i, res in enumerate(res_list):
    sns.scatterplot(x=res[:,0], y=res[:,1], hue=encoder.inverse_transform(y), ax=axes[i], palette='Set1')
    axes[i].set_title(titles[i])

plt.tight_layout()
plt.savefig(PLOT_DIR / "dimensionality_reduction.png", dpi=300)
plt.show()

### STEP 9: Biomarker Statistical Validation
We perform Mann-Whitney U tests and generate distribution plots for the leading ECG biomarkers.

In [ ]:
biomarker_stats = []
key_biomarkers = confirmed_ecg[:3] # HRV SDNN, P-Wave Dispersion, SDNN/MeanRR

fig, axes = plt.subplots(len(key_biomarkers), 2, figsize=(12, 4 * len(key_biomarkers)))

for i, col in enumerate(key_biomarkers):
    # Stats
    group_0 = X_filtered[y == 0][col]
    group_1 = X_filtered[y == 1][col]
    stat, p_val = mannwhitneyu(group_0, group_1)
    biomarker_stats.append({'Feature': col, 'p-value': p_val, 'U-statistic': stat})

    # Plots
    sns.boxplot(x=encoder.inverse_transform(y), y=X_filtered[col], ax=axes[i, 0], palette='pastel')
    sns.violinplot(x=encoder.inverse_transform(y), y=X_filtered[col], ax=axes[i, 1], palette='pastel')
    axes[i, 0].set_title(f'{col} Boxplot')
    axes[i, 1].set_title(f'{col} Violin plot')

plt.tight_layout()
plt.savefig(PLOT_DIR / "biomarker_distributions.png", dpi=300)
plt.show()

stats_df = pd.DataFrame(biomarker_stats)
stats_df.to_csv(TABLE_DIR / "biomarker_consistency_report.csv", index=False)
display(stats_df)

### STEP 10: Final Reports & Export
Saving the correlation matrix and generating the final executive summary.

In [ ]:
X.corr().to_csv(TABLE_DIR / "correlation_matrix.csv")
comparison_extended.to_csv(TABLE_DIR / "feature_group_comparison.csv", index=False)

summary = f"""
EXECUTIVE SUMMARY
-----------------
Top ECG Biomarkers: {', '.join(confirmed_ecg)}
Top Phys Biomarkers: {', '.join(phys_features)}

ECG-only AUC: {comparison_df.iloc[0]['AUC']:.3f}
Phys-only AUC: {comparison_df.iloc[1]['AUC']:.3f}
Combined AUC: {comparison_df.iloc[2]['AUC']:.3f}
Weighted Ensemble AUC: {weighted_results['AUC']:.3f}

Conclusion: READY FOR MODEL TRAINING
"""

with open(REPORT_DIR / "executive_summary.txt", "w") as f:
    f.write(summary)

print(summary)

### STEP 11: Feature Validation (RF Importance vs Permutation Importance)
As requested, we compare the Boruta ranking with standard Random Forest importance and Permutation Importance to ensure the biomarkers are truly significant.

In [ ]:
from sklearn.inspection import permutation_importance

# 1. RF Feature Importance
rf_temp = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
rf_temp.fit(X_filtered, y)
rf_imp = rf_temp.feature_importances_

# 2. Permutation Importance
perm_imp = permutation_importance(rf_temp, X_filtered, y, n_repeats=10, random_state=RANDOM_SEED)

# 3. Create Feature Validation Table
feature_val_df = pd.DataFrame({
    'Feature': X_filtered.columns,
    'Boruta_Status': boruta_results.set_index('Feature').reindex(X_filtered.columns)['Status'].values,
    'RF_Importance': rf_imp,
    'Permutation_Importance': perm_imp.importances_mean
}).sort_values('Permutation_Importance', ascending=False)

feature_val_df.to_csv(TABLE_DIR / "feature_validation_table.csv", index=False)
display(feature_val_df)

### STEP 12: Publication Quality Visualizations
Saving high-resolution plots for Correlation, MI, and Boruta as requested.

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(X.corr(), annot=False, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.savefig(PLOT_DIR / "correlation_heatmap.png", dpi=300)
plt.close()

# MI Ranking Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=mi_df, x='MI_Score', y='Feature', palette='magma')
plt.title('Mutual Information Ranking')
plt.savefig(PLOT_DIR / "mutual_information_ranking.png", dpi=300)
plt.close()

# Boruta Status Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=boruta_results, x='Ranking', y='Feature', hue='Status')
plt.title('Boruta Feature Selection Status')
plt.savefig(PLOT_DIR / "boruta_status_plot.png", dpi=300)
plt.close()

print("Publication quality plots saved to /outputs/plots/")

### STEP 13: Objective Assessment Report
Final status report for the project objectives.

In [ ]:
# Google Drive mount removed for publication.
# Place the input CSV next to this notebook; it is read with a relative path.

In [ ]:
objectives = [
    {"Objective": "Create ECG-centric PCOS dataset", "Status": "ACHIEVED"},
    {"Objective": "Identify ECG biomarkers", "Status": "ACHIEVED"},
    {"Objective": "Obtain final feature subset", "Status": "ACHIEVED"},
    {"Objective": "Ready for model training", "Status": "READY"}
]
obj_report = pd.DataFrame(objectives)
obj_report.to_csv(REPORT_DIR / "objective_assessment_report.csv", index=False)

display(obj_report)
print("\nFINAL RECOMMENDATION: READY FOR MODEL TRAINING")

### STEP 11: Feature Validation (RF Importance vs Permutation Importance)
As requested, we compare the Boruta ranking with standard Random Forest importance and Permutation Importance to ensure the biomarkers are truly significant.

In [ ]:
from sklearn.inspection import permutation_importance

# 1. RF Feature Importance
rf_temp = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
rf_temp.fit(X_filtered, y)
rf_imp = rf_temp.feature_importances_

# 2. Permutation Importance
perm_imp = permutation_importance(rf_temp, X_filtered, y, n_repeats=10, random_state=RANDOM_SEED)

# 3. Create Feature Validation Table
feature_val_df = pd.DataFrame({
    'Feature': X_filtered.columns,
    'Boruta_Status': boruta_results.set_index('Feature').reindex(X_filtered.columns)['Status'].values,
    'RF_Importance': rf_imp,
    'Permutation_Importance': perm_imp.importances_mean
}).sort_values('Permutation_Importance', ascending=False)

feature_val_df.to_csv(TABLE_DIR / "feature_validation_table.csv", index=False)
display(feature_val_df)

### STEP 12: Publication Quality Visualizations
Saving high-resolution plots for Correlation, MI, and Boruta as requested.

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(X.corr(), annot=False, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.savefig(PLOT_DIR / "correlation_heatmap.png", dpi=300)
plt.close()

# MI Ranking Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=mi_df, x='MI_Score', y='Feature', palette='magma')
plt.title('Mutual Information Ranking')
plt.savefig(PLOT_DIR / "mutual_information_ranking.png", dpi=300)
plt.close()

# Boruta Status Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=boruta_results, x='Ranking', y='Feature', hue='Status')
plt.title('Boruta Feature Selection Status')
plt.savefig(PLOT_DIR / "boruta_status_plot.png", dpi=300)
plt.close()

print("Publication quality plots saved to /outputs/plots/")

### STEP 13: Objective Assessment Report
Final status report for the project objectives.

In [ ]:
objectives = [
    {"Objective": "Create ECG-centric PCOS dataset", "Status": "ACHIEVED"},
    {"Objective": "Identify ECG biomarkers", "Status": "ACHIEVED"},
    {"Objective": "Obtain final feature subset", "Status": "ACHIEVED"},
    {"Objective": "Ready for model training", "Status": "READY"}
]
obj_report = pd.DataFrame(objectives)
obj_report.to_csv(REPORT_DIR / "objective_assessment_report.csv", index=False)

display(obj_report)
print("\nFINAL RECOMMENDATION: READY FOR MODEL TRAINING")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Remove Subject_ID and filter X
subject_id_cols = [c for c in X.columns if 'subject_id' in c.lower()]
X_clean = X.drop(columns=subject_id_cols)

# 2. Rerun Correlation Filtering
corr_matrix_clean = X_clean.corr().abs()
upper_clean = corr_matrix_clean.where(np.triu(np.ones(corr_matrix_clean.shape), k=1).astype(bool))
to_drop_clean = [column for column in upper_clean.columns if any(upper_clean[column] > 0.90)]
X_final = X_clean.drop(columns=to_drop_clean)

# 3. Rerun MI and Boruta
mi_scores_final = mutual_info_classif(X_final, y, random_state=RANDOM_SEED)
rf_boruta = RandomForestClassifier(n_jobs=-1, class_weight='balanced', max_depth=5, random_state=RANDOM_SEED)
boruta_selector_final = BorutaPy(rf_boruta, n_estimators='auto', random_state=RANDOM_SEED)
boruta_selector_final.fit(X_final.values, y)

# 4. Generate the Definitive Feature List
feature_list_data = []
for i, col in enumerate(X_final.columns):
    status = 'Keep' if boruta_selector_final.support_[i] or boruta_selector_final.support_weak_[i] else 'Reject'
    reason = 'Confirmed/Tentative by Boruta' if status == 'Keep' else 'Insufficient predictive power'
    feature_list_data.append({'Feature': col, 'Keep/Reject': status, 'Reason': reason})

# Add dropped columns to the list for completeness
for col in to_drop_clean:
    feature_list_data.append({'Feature': col, 'Keep/Reject': 'Reject', 'Reason': 'Multicollinearity (|r| > 0.90)'})
for col in subject_id_cols:
    feature_list_data.append({'Feature': col, 'Keep/Reject': 'Reject', 'Reason': 'Identifier (Non-biological)'})

final_features_df = pd.DataFrame(feature_list_data)
final_features_df.to_csv(TABLE_DIR / 'final_selected_features.csv', index=False)
display(final_features_df)

In [ ]:
# 5. Export Separate High-Res Figures for PCA, UMAP, t-SNE
projections = {'PCA': pca_res, 'UMAP': umap_res, 't-SNE': tsne_res}

for name, data in projections.items():
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=data[:, 0], y=data[:, 1], hue=encoder.inverse_transform(y), palette='Set1', s=60, alpha=0.8)
    plt.title(f'{name} Projection of PCOS Biomarkers', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.savefig(PLOT_DIR / f'final_{name.lower()}_300dpi.png', dpi=300, bbox_inches='tight')
    plt.show()

print(f"Individual figures saved to {PLOT_DIR}")

In [ ]:
# 6. Update the final archive
shutil.make_archive('project_results_updated', 'zip', 'outputs')
files.download('project_results_updated.zip')
print("Final updated archive generated.")

In [ ]:
import shutil
from google.colab import files

# Compress the outputs folder
shutil.make_archive('project_results', 'zip', 'outputs')
print('Project results compressed successfully as project_results.zip')

# Trigger the browser download
files.download('project_results.zip')